# SAM3 Auto Dataset Factory — Colab GPU Server

Hücreleri **sırayla** çalıştırın. Son hücre size bir `ngrok` URL verecek — bunu web uygulamasının **Settings → API Server URL** alanına yapıştırın.

> **Önce:** Runtime → Change runtime type → **T4 GPU** seçin (ücretsiz)

In [ ]:
# 1 — GPU kontrolü
import subprocess, sys
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'Bulunamadi — T4 GPU secin')
print('Python:', sys.version)

In [ ]:
# 2 — Paket kurulumu
!pip install -q fastapi 'uvicorn[standard]' pyngrok python-multipart pillow aiofiles
print('Kurulum tamamlandi.')

In [ ]:
# 3 — Ayarlar
# USE_REAL_MODEL = False -> mock adapter (GPU gerekmez, hizli)
# USE_REAL_MODEL = True  -> Grounding DINO (T4 GPU gerekir, gercek detection)
USE_REAL_MODEL = False
API_KEY = 'sam3-test-2024'   # Istediginiz bir key girebilirsiniz
PORT    = 8000

import os
os.environ['API_KEY']   = API_KEY
os.environ['USE_MOCK']  = 'false' if USE_REAL_MODEL else 'true'
os.environ['DEVICE']    = 'cuda:0' if USE_REAL_MODEL else 'cpu'
os.environ['PORT']      = str(PORT)

if USE_REAL_MODEL:
    print('Gercek model kurulumu basliyor...')
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
    !pip install -q transformers huggingface_hub
    print('Gercek model paketleri kuruldu.')
else:
    print('Mock mode aktif — GPU gerekmez.')

print(f'API Key: {API_KEY}  |  Port: {PORT}  |  Mock: {not USE_REAL_MODEL}')

In [ ]:
%%writefile /content/server_part1.py
# --- Imports & config ---
import os, uuid, random, threading
from datetime import datetime
from pathlib import Path
from typing import Optional
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException, UploadFile, File, BackgroundTasks, Depends, Security
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, FileResponse
from fastapi.security.api_key import APIKeyHeader

API_KEY  = os.environ.get('API_KEY', 'change-me')
DEVICE   = os.environ.get('DEVICE', 'cpu')
USE_MOCK = os.environ.get('USE_MOCK', 'true').lower() == 'true'

api_key_header = APIKeyHeader(name='X-API-Key', auto_error=False)

async def verify_key(key: str | None = Security(api_key_header)):
    if API_KEY == 'change-me':
        return
    if key != API_KEY:
        raise HTTPException(status_code=403, detail='Invalid API key')

def new_id(): return str(uuid.uuid4())
def now():    return datetime.utcnow()

# --- In-memory store ---
datasets    = {}
images      = {}
jobs        = {}
annotations = {}
exports     = {}
export_data = {}

def seed_data():
    ds_id = new_id()
    ts    = now()
    datasets[ds_id] = {
        'id': ds_id, 'name': 'Colab Test Dataset',
        'description': 'Colab sunucusunda olusturuldu',
        'classes': [{'id': 0, 'name': 'object', 'prompt': 'a salient object', 'color': '#ef4444'}],
        'image_count': 0, 'annotated_count': 0,
        'train_split': 0.8, 'val_split': 0.1, 'test_split': 0.1,
        'output_mode': 'bbox_and_segmentation', 'prompt_type': 'text',
        'created_at': ts, 'updated_at': ts,
    }

seed_data()
print('In-memory store hazir.')

In [ ]:
%%writefile /content/server_part2.py
# --- Adapters ---

class MockAdapter:
    model_loaded = True
    available_memory_gb = None

    def predict(self, width, height, class_prompts, confidence_threshold=0.35, max_detections=50):
        n = random.randint(1, min(5, max_detections))
        results = []
        for _ in range(n):
            cls  = random.choice(class_prompts)
            x1   = random.uniform(0, width  * 0.7)
            y1   = random.uniform(0, height * 0.7)
            x2   = x1 + random.uniform(width  * 0.1, min(width  * 0.4, width  - x1))
            y2   = y1 + random.uniform(height * 0.1, min(height * 0.4, height - y1))
            conf = random.uniform(0.45, 0.98)
            if conf < confidence_threshold:
                continue
            results.append({
                'class_id':   cls['id'],
                'class_name': cls['name'],
                'confidence': conf,
                'bbox_xyxy':  [x1, y1, x2, y2],
            })
        return results


class RealAdapter:
    def __init__(self):
        self.model_loaded       = False
        self.available_memory_gb = None
        self._model     = None
        self._processor = None

    def load(self):
        import torch
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        model_id = 'IDEA-Research/grounding-dino-base'
        print(f'Loading {model_id} on {DEVICE}...')
        self._processor = AutoProcessor.from_pretrained(model_id)
        self._model     = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(DEVICE)
        self.model_loaded = True
        if torch.cuda.is_available():
            self.available_memory_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
        print('Model yuklendi!')

    def predict(self, width, height, class_prompts, confidence_threshold=0.35, max_detections=50):
        import torch
        from PIL import Image as PILImage
        if not self.model_loaded:
            self.load()
        text   = ' . '.join(c['prompt'] for c in class_prompts) + '.'
        dummy  = PILImage.new('RGB', (width, height), (128, 128, 128))
        inputs = self._processor(images=dummy, text=text, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            outputs = self._model(**inputs)
        results = self._processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            threshold=confidence_threshold,
            text_threshold=confidence_threshold * 0.7,
            target_sizes=[(height, width)]
        )[0]
        detections = []
        for score, label_id, box in zip(results['scores'], results['labels'], results['boxes']):
            if len(detections) >= max_detections:
                break
            idx = label_id.item() % len(class_prompts)
            detections.append({
                'class_id':   class_prompts[idx]['id'],
                'class_name': class_prompts[idx]['name'],
                'confidence': float(score),
                'bbox_xyxy':  [float(v) for v in box],
            })
        return detections


adapter = MockAdapter() if USE_MOCK else RealAdapter()

In [ ]:
%%writefile /content/server_part3.py
# --- Job runner ---

def run_job(job_id):
    job = jobs.get(job_id)
    if not job:
        return
    job['status'] = 'running'
    job['started_at'] = now()
    class_prompts = [{'id': c['class_id'], 'name': c['class_name'], 'prompt': c['prompt']}
                     for c in job['classes']]
    ds_imgs   = [i for i in images.values() if i['dataset_id'] == job['dataset_id']]
    job['total_images'] = max(len(ds_imgs), job['total_images'])
    processed = 0
    failed    = 0
    for img in ds_imgs:
        if jobs.get(job_id, {}).get('status') == 'cancelled':
            return
        try:
            dets = adapter.predict(
                img.get('width', 640), img.get('height', 480),
                class_prompts,
                job['config'].get('confidence_threshold', 0.35),
                job['config'].get('max_detections_per_image', 50),
            )
            for det in dets:
                aid = new_id()
                annotations[aid] = {
                    'id': aid, 'image_id': img['id'],
                    'class_id': det['class_id'], 'class_name': det['class_name'],
                    'confidence': det['confidence'], 'bbox_xyxy': det['bbox_xyxy'],
                    'polygon': None, 'mask_area': None, 'point_count': None,
                    'review_status': 'pending', 'source': 'auto', 'created_at': now(),
                }
            img['annotation_count'] = sum(1 for a in annotations.values()
                                          if a['image_id'] == img['id'])
            processed += 1
        except Exception as e:
            print(f'Image {img["id"]} failed: {e}')
            failed += 1
        job['processed_images'] = processed
        job['failed_images']    = failed
        job['progress']         = processed / max(job['total_images'], 1)
    job['status']       = 'completed'
    job['progress']     = 1.0
    job['completed_at'] = now()
    ds = datasets.get(job['dataset_id'])
    if ds:
        ds['annotated_count'] = min(ds['annotated_count'] + processed, ds['image_count'])
        ds['updated_at']      = now()

In [ ]:
%%writefile /content/server_part4.py
# --- FastAPI app & routes ---

@asynccontextmanager
async def lifespan(app):
    Path('/content/data').mkdir(exist_ok=True)
    if not USE_MOCK:
        threading.Thread(target=adapter.load, daemon=True).start()
    yield

app = FastAPI(
    title='SAM3 Auto Dataset Factory',
    version='0.1.0',
    lifespan=lifespan,
    docs_url='/api/docs',
    redoc_url=None,
    openapi_url='/api/openapi.json',
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)
auth = [Depends(verify_key)]

def fmt_ds(d):  return {**d, 'created_at': d['created_at'].isoformat(), 'updated_at': d['updated_at'].isoformat()}
def fmt_img(i): return {**i, 'created_at': i['created_at'].isoformat()}
def fmt_ann(a): return {**a, 'created_at': a['created_at'].isoformat()}
def fmt_job(j): return {**j,
    'created_at':   j['created_at'].isoformat(),
    'started_at':   j['started_at'].isoformat()   if j['started_at']   else None,
    'completed_at': j['completed_at'].isoformat() if j['completed_at'] else None,
}
def fmt_exp(e): return {**e,
    'created_at':   e['created_at'].isoformat(),
    'completed_at': e['completed_at'].isoformat() if e['completed_at'] else None,
}

@app.get('/api/healthz')
async def health():
    return {'status': 'ok'}

@app.get('/api/server-info', dependencies=auth)
async def server_info():
    return {
        'status': 'ok', 'device': DEVICE,
        'model_loaded': adapter.model_loaded,
        'model_name': 'grounding-dino-base' if not USE_MOCK else 'sam3-mock',
        'max_batch_size': 8,
        'available_memory_gb': adapter.available_memory_gb,
        'mock_mode': USE_MOCK,
    }

@app.get('/api/stats/dashboard', dependencies=auth)
async def stats():
    return {
        'total_datasets':    len(datasets),
        'total_images':      len(images),
        'total_annotations': len(annotations),
        'pending_review':    sum(1 for a in annotations.values() if a['review_status'] == 'pending'),
        'completed_jobs':    sum(1 for j in jobs.values() if j['status'] == 'completed'),
        'running_jobs':      sum(1 for j in jobs.values() if j['status'] == 'running'),
        'recent_exports':    len(exports),
        'server_status':     'ok',
        'model_loaded':      adapter.model_loaded,
    }

@app.get('/api/datasets', dependencies=auth)
async def list_datasets(page: int = 1, limit: int = 20):
    all_ds = sorted(datasets.values(), key=lambda d: d['created_at'], reverse=True)
    start  = (page - 1) * limit
    return {'items': [fmt_ds(d) for d in all_ds[start:start+limit]], 'total': len(all_ds), 'page': page, 'limit': limit}

@app.post('/api/datasets', dependencies=auth, status_code=201)
async def create_dataset(body: dict):
    ts = now(); ds_id = new_id()
    ds = {
        'id': ds_id, 'name': body['name'], 'description': body.get('description'),
        'classes': body['classes'], 'image_count': 0, 'annotated_count': 0,
        'train_split': body.get('train_split', 0.8), 'val_split': body.get('val_split', 0.1),
        'test_split': body.get('test_split', 0.1),
        'output_mode': body.get('output_mode', 'bbox_and_segmentation'),
        'prompt_type': body.get('prompt_type', 'text'),
        'created_at': ts, 'updated_at': ts,
    }
    datasets[ds_id] = ds
    return fmt_ds(ds)

@app.get('/api/datasets/{dataset_id}', dependencies=auth)
async def get_dataset(dataset_id: str):
    ds = datasets.get(dataset_id)
    if not ds: raise HTTPException(404, 'Not found')
    return fmt_ds(ds)

@app.delete('/api/datasets/{dataset_id}', dependencies=auth, status_code=204)
async def delete_dataset(dataset_id: str):
    if dataset_id not in datasets: raise HTTPException(404, 'Not found')
    del datasets[dataset_id]

@app.get('/api/datasets/{dataset_id}/images', dependencies=auth)
async def list_images(dataset_id: str, split: str = 'all', page: int = 1, limit: int = 50):
    if dataset_id not in datasets: raise HTTPException(404, 'Not found')
    imgs = [i for i in images.values() if i['dataset_id'] == dataset_id]
    if split != 'all': imgs = [i for i in imgs if i['split'] == split]
    start = (page - 1) * limit
    return {'items': [fmt_img(i) for i in imgs[start:start+limit]], 'total': len(imgs), 'page': page, 'limit': limit}

@app.post('/api/datasets/{dataset_id}/upload', dependencies=auth)
async def upload_images(dataset_id: str, files: list[UploadFile] = File(...)):
    ds = datasets.get(dataset_id)
    if not ds: raise HTTPException(404, 'Not found')
    splits    = ['train', 'val', 'test']
    uploaded  = 0
    errors    = []
    save_dir  = Path('/content/data') / dataset_id
    save_dir.mkdir(parents=True, exist_ok=True)
    for idx, f in enumerate(files):
        try:
            content = await f.read()
            from PIL import Image as PILImage
            import io
            pil = PILImage.open(io.BytesIO(content))
            w, h = pil.size
            fname = f.filename or f'img_{new_id()}.jpg'
            (save_dir / fname).write_bytes(content)
            img_id = new_id()
            images[img_id] = {
                'id': img_id, 'dataset_id': dataset_id,
                'filename': fname, 'width': w, 'height': h,
                'split': splits[idx % 3],
                'annotation_count': 0, 'review_status': 'pending',
                'url': f'/api/datasets/{dataset_id}/images/{img_id}/file',
                'created_at': now(),
            }
            uploaded += 1
        except Exception as e:
            errors.append(str(e))
    ds['image_count'] += uploaded
    ds['updated_at']   = now()
    return {'uploaded': uploaded, 'failed': len(errors), 'errors': errors}

@app.get('/api/datasets/{dataset_id}/images/{image_id}/file', dependencies=auth)
async def serve_image(dataset_id: str, image_id: str):
    img = images.get(image_id)
    if not img: raise HTTPException(404, 'Not found')
    p = Path('/content/data') / dataset_id / img['filename']
    if not p.exists(): raise HTTPException(404, 'File not found')
    return FileResponse(p)

@app.get('/api/jobs', dependencies=auth)
async def list_jobs(status: Optional[str] = None, page: int = 1, limit: int = 20):
    all_j = sorted(jobs.values(), key=lambda j: j['created_at'], reverse=True)
    if status: all_j = [j for j in all_j if j['status'] == status]
    start = (page - 1) * limit
    return {'items': [fmt_job(j) for j in all_j[start:start+limit]], 'total': len(all_j), 'page': page, 'limit': limit}

@app.post('/api/jobs/auto-label', dependencies=auth, status_code=201)
async def create_job(body: dict, background_tasks: BackgroundTasks):
    if body['dataset_id'] not in datasets: raise HTTPException(404, 'Dataset not found')
    ds_imgs = [i for i in images.values() if i['dataset_id'] == body['dataset_id']]
    j_id = new_id()
    job  = {
        'id': j_id, 'dataset_id': body['dataset_id'], 'status': 'queued',
        'progress': 0.0, 'processed_images': 0,
        'total_images': len(ds_imgs) or datasets[body['dataset_id']]['image_count'],
        'failed_images': 0,
        'classes': body['classes'], 'config': body['config'],
        'error_message': None,
        'created_at': now(), 'started_at': None, 'completed_at': None,
    }
    jobs[j_id] = job
    background_tasks.add_task(lambda: threading.Thread(target=run_job, args=(j_id,), daemon=True).start())
    return fmt_job(job)

@app.get('/api/jobs/{job_id}', dependencies=auth)
async def get_job(job_id: str):
    job = jobs.get(job_id)
    if not job: raise HTTPException(404, 'Not found')
    return fmt_job(job)

@app.post('/api/jobs/{job_id}/cancel', dependencies=auth)
async def cancel_job(job_id: str):
    job = jobs.get(job_id)
    if not job: raise HTTPException(404, 'Not found')
    if job['status'] in ('running', 'queued'):
        job['status'] = 'cancelled'
        job['completed_at'] = now()
    return fmt_job(job)

@app.get('/api/images/{image_id}/annotations', dependencies=auth)
async def list_annotations(image_id: str, minConfidence: float = 0.0):
    return [fmt_ann(a) for a in annotations.values()
            if a['image_id'] == image_id and a['confidence'] >= minConfidence]

@app.post('/api/images/{image_id}/annotations', dependencies=auth, status_code=201)
async def create_annotation(image_id: str, body: dict):
    if image_id not in images: raise HTTPException(404, 'Not found')
    ann_id = new_id()
    ann = {
        'id': ann_id, 'image_id': image_id,
        'class_id': body['class_id'], 'class_name': body.get('class_name', ''),
        'confidence': 1.0, 'bbox_xyxy': body['bbox_xyxy'],
        'polygon': body.get('polygon'), 'mask_area': None, 'point_count': None,
        'review_status': 'accepted', 'source': 'manual', 'created_at': now(),
    }
    annotations[ann_id] = ann
    return fmt_ann(ann)

@app.put('/api/annotations/{ann_id}', dependencies=auth)
async def update_annotation(ann_id: str, body: dict):
    ann = annotations.get(ann_id)
    if not ann: raise HTTPException(404, 'Not found')
    for k, v in body.items():
        if v is not None: ann[k] = v
    return fmt_ann(ann)

@app.delete('/api/annotations/{ann_id}', dependencies=auth, status_code=204)
async def delete_annotation(ann_id: str):
    if ann_id not in annotations: raise HTTPException(404, 'Not found')
    del annotations[ann_id]

@app.post('/api/images/{image_id}/reprocess', dependencies=auth)
async def reprocess_image(image_id: str, body: dict, background_tasks: BackgroundTasks):
    img = images.get(image_id)
    if not img: raise HTTPException(404, 'Not found')
    to_del = [a['id'] for a in annotations.values()
              if a['image_id'] == image_id and a['source'] == 'auto']
    for aid in to_del:
        del annotations[aid]
    def _reproc():
        dets = adapter.predict(
            img.get('width', 640), img.get('height', 480),
            [{'id': c['class_id'], 'name': c['class_name'], 'prompt': c['prompt']}
             for c in body['classes']],
            body.get('confidence_threshold', 0.35),
        )
        for det in dets:
            aid = new_id()
            annotations[aid] = {
                'id': aid, 'image_id': image_id,
                'class_id': det['class_id'], 'class_name': det['class_name'],
                'confidence': det['confidence'], 'bbox_xyxy': det['bbox_xyxy'],
                'polygon': None, 'mask_area': None, 'point_count': None,
                'review_status': 'pending', 'source': 'auto', 'created_at': now(),
            }
        img['annotation_count'] = sum(1 for a in annotations.values() if a['image_id'] == image_id)
    background_tasks.add_task(lambda: threading.Thread(target=_reproc, daemon=True).start())
    return {'image_id': image_id, 'status': 'reprocessing', 'annotation_count': img['annotation_count']}

In [ ]:
%%writefile /content/server_part5.py
# --- Export routes ---
import io, zipfile

@app.post('/api/exports', dependencies=auth, status_code=201)
async def create_export(body: dict, background_tasks: BackgroundTasks):
    if body['dataset_id'] not in datasets: raise HTTPException(404, 'Not found')
    exp_id = new_id()
    exp = {
        'id': exp_id, 'dataset_id': body['dataset_id'],
        'format': body['format'], 'status': 'pending',
        'download_url': None, 'file_size_bytes': None,
        'image_count': sum(1 for i in images.values() if i['dataset_id'] == body['dataset_id']),
        'annotation_count': 0,
        'min_confidence': body.get('min_confidence', 0.0),
        'include_rejected': body.get('include_rejected', False),
        'created_at': now(), 'completed_at': None,
    }
    exports[exp_id] = exp

    def _build():
        exp['status'] = 'running'
        buf = io.BytesIO()
        min_conf     = body.get('min_confidence', 0.0)
        inc_rejected = body.get('include_rejected', False)
        classes      = datasets[body['dataset_id']]['classes']
        with zipfile.ZipFile(buf, 'w') as zf:
            names = '\n'.join(f'  {c["id"]}: {c["name"]}' for c in classes)
            yaml  = f'path: .\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(classes)}\nnames:\n{names}\n'
            zf.writestr('data.yaml', yaml)
            ds_imgs   = [i for i in images.values() if i['dataset_id'] == body['dataset_id']]
            ann_count = 0
            for img in ds_imgs:
                split = img['split']
                img_path = Path('/content/data') / body['dataset_id'] / img['filename']
                if img_path.exists():
                    zf.write(str(img_path), f'images/{split}/{img["filename"]}')
                else:
                    zf.writestr(f'images/{split}/{img["filename"]}', b'')
                img_anns = [a for a in annotations.values()
                            if a['image_id'] == img['id']
                            and a['confidence'] >= min_conf
                            and (inc_rejected or a['review_status'] != 'rejected')]
                lines = []
                for a in img_anns:
                    x1, y1, x2, y2 = a['bbox_xyxy']
                    w = img.get('width',  640)
                    h = img.get('height', 480)
                    cx = (x1 + x2) / 2 / w
                    cy = (y1 + y2) / 2 / h
                    bw = (x2 - x1) / w
                    bh = (y2 - y1) / h
                    lines.append(f'{a["class_id"]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                    ann_count += 1
                stem = Path(img['filename']).stem
                zf.writestr(f'labels/{split}/{stem}.txt', '\n'.join(lines))
        zip_bytes = buf.getvalue()
        export_data[exp_id] = zip_bytes
        exp['status']        = 'completed'
        exp['completed_at']  = now()
        exp['download_url']  = f'/api/exports/{exp_id}/download'
        exp['file_size_bytes'] = len(zip_bytes)
        exp['annotation_count'] = ann_count

    background_tasks.add_task(lambda: threading.Thread(target=_build, daemon=True).start())
    return fmt_exp(exp)

@app.get('/api/exports', dependencies=auth)
async def list_exports(datasetId: Optional[str] = None):
    exps = sorted(exports.values(), key=lambda e: e['created_at'], reverse=True)
    if datasetId: exps = [e for e in exps if e['dataset_id'] == datasetId]
    return [fmt_exp(e) for e in exps]

@app.get('/api/exports/{exp_id}', dependencies=auth)
async def get_export(exp_id: str):
    e = exports.get(exp_id)
    if not e: raise HTTPException(404, 'Not found')
    return fmt_exp(e)

@app.get('/api/exports/{exp_id}/download', dependencies=auth)
async def download_export(exp_id: str):
    e = exports.get(exp_id)
    if not e or e['status'] != 'completed': raise HTTPException(404, 'Not ready')
    b = export_data.get(exp_id)
    if not b: raise HTTPException(404, 'File not found')
    return Response(
        content=b,
        media_type='application/zip',
        headers={'Content-Disposition': 'attachment; filename=dataset_yolo.zip'},
    )

In [ ]:
# 4 — Dosyaları birlestir -> server.py
parts = ['/content/server_part1.py', '/content/server_part2.py',
         '/content/server_part3.py', '/content/server_part4.py',
         '/content/server_part5.py']
with open('/content/server.py', 'w') as out:
    for p in parts:
        with open(p) as f:
            out.write(f.read())
        out.write('\n')
print('server.py olusturuldu —', sum(1 for _ in open('/content/server.py')), 'satir')

In [ ]:
# 5 — ngrok token (https://dashboard.ngrok.com/get-started/your-authtoken)
from pyngrok import ngrok

NGROK_TOKEN = ''  # <-- tokeninizi buraya yapistirin

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print('ngrok token ayarlandi.')
else:
    print('UYARI: Token bos. https://dashboard.ngrok.com/get-started/your-authtoken adresinden alin.')

In [ ]:
# 6 — Sunucuyu baslat + ngrok tunnel ac
import subprocess, time

try: ngrok.kill()
except: pass

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', str(PORT)],
    cwd='/content',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print('Sunucu baslatiliyor...')
time.sleep(5)

try:
    tunnel     = ngrok.connect(PORT, 'http')
    public_url = tunnel.public_url.replace('http://', 'https://')

    sep = '=' * 62
    print(sep)
    print('SUNUCU HAZIR!')
    print(sep)
    print(f'Public URL : {public_url}')
    print(f'API Key    : {API_KEY}')
    print(f'API Docs   : {public_url}/api/docs')
    print(sep)
    print('Web uygulamasinda:')
    print(f'  Settings -> API Server URL : {public_url}')
    print(f'  Settings -> API Key        : {API_KEY}')
    print('  Save & Reload tiklayın')
    print(sep)
except Exception as err:
    print(f'ngrok hatasi: {err}')
    print('Token ekleyip Hucre 5 ve 6yi tekrar calistirin.')

In [ ]:
# 7 — Hizli saglik kontrolu
import urllib.request, json

def api_get(path):
    req = urllib.request.Request(
        f'http://localhost:{PORT}{path}',
        headers={'X-API-Key': API_KEY},
    )
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())

print('health     :', api_get('/api/healthz'))
print('server-info:', api_get('/api/server-info'))
print('dashboard  :', api_get('/api/stats/dashboard'))

In [ ]:
# 8 — Sunucu loglarini goster (istenirse)
for _ in range(30):
    line = proc.stdout.readline()
    if not line: break
    print(line.decode().rstrip())